# Survival Simulation Demo Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vandrearczyk/survival-simulation/blob/main/demo.ipynb)

This notebook demonstrates how to use `survival-simulation` via both **PyPI** (Python API) and **Docker** (CLI).

## 1. PyPI Usage (Python API)

Install `survival-simulation` directly from PyPI.

In [ ]:
# Install the published PyPI package
!pip install survival-simulation lifelines matplotlib pandas

In [ ]:
import pandas as pd
from survival_simulation import simulate_survival

# Generate data in the 'linear' scenario returning Pandas DataFrames
X, y_censored, y_true = simulate_survival(
    n_samples=1000,
    n_features=20,
    scenario="linear",
    censoring=0.3,
    seed=42,
    return_type="pandas"
)

print(f"Feature matrix shape: {X.shape}")
print("\nCensored target sample:")
print(y_censored.head())
print(f"\nObserved censoring rate: {1 - y_censored['event'].mean():.2%}")

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

# Compare Kaplan-Meier survival curves across all scenarios
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
scenarios = ["linear", "nonlinear", "non_ph"]
kmf = KaplanMeierFitter()

for ax, scenario in zip(axes, scenarios):
    X, y_censored, _ = simulate_survival(
        n_samples=1000,
        scenario=scenario,
        censoring=0.3,
        seed=42,
        return_type="pandas"
    )
    
    kmf.fit(y_censored["time"], event_observed=y_censored["event"], label=scenario)
    kmf.plot_survival_function(ax=ax)
    ax.set_title(f"Scenario: '{scenario}'")
    ax.set_xlabel("Time")
    ax.set_ylabel("Survival Probability")
    ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 2. Docker Usage (CLI)

Pull and execute the published Docker image from Docker Hub to generate datasets without Python dependencies.

In [ ]:
# Create a output directory
!mkdir -p data

# Pull the image from Docker Hub
!docker pull vandrearczyk/survival-simulation:latest

# Run the container CLI to generate a CSV dataset
!docker run --rm -v $(pwd)/data:/data vandrearczyk/survival-simulation \
  --scenario non_ph \
  --n-samples 500 \
  --censoring 0.25 \
  --out /data/docker_output.csv

In [ ]:
# Inspect the dataset generated by Docker
df_docker = pd.read_csv("data/docker_output.csv")
print(f"Generated dataset shape: {df_docker.shape}")
df_docker.head()